# Pakistan House Price Prediction: Six Regression Algorithms Compared
**Dataset:** Zameen.com listings (Karachi, Lahore, Islamabad, Rawalpindi, Faisalabad) · **Target:** price in PKR

Algorithms: Linear Regression · Decision Tree · Random Forest · Neural Network (MLP) · XGBoost · KNN, all trained on the
**same cleaned data, the same 14 inputs and the same 80:20 split**, and evaluated with **MAE, MSE, RMSE, R²** (plus MAPE).

The full illustrated report is `REPORT.md`. Figures are loaded from `results/figures/` (run `python train.py` first if they are missing).

In [1]:
import warnings, time, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from src.data_preprocessing import load_raw, clean, add_features, get_train_test, build_preprocessor, FEATURES, TARGET, format_pkr
from src.models import get_models, MODEL_ORDER
from src.evaluate import regression_metrics
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 30)

## 1. Load the raw Zameen.com data

In [2]:
raw = load_raw()            # data/raw/zameen-updated.csv.xz
print(raw.shape)
raw.head(3).T

(168446, 20)


,0,1,2
property_id,237062,346905,386513
location_id,3325,3236,764
page_url,https://www.zameen.com/Property/g_10_g_10_2_gr...,https://www.zameen.com/Property/e_11_2_service...,https://www.zameen.com/Property/islamabad_g_15...
property_type,Flat,Flat,House
price,10000000,6900000,16500000
location,G-10,E-11,G-15
city,Islamabad,Islamabad,Islamabad
province_name,Islamabad Capital,Islamabad Capital,Islamabad Capital
latitude,33.6799,33.7010,33.6315
longitude,73.0126,72.9715,72.9266


In [3]:
raw.isna().sum()

property_id          0
location_id          0
page_url             0
property_type        0
price                0
location             0
city                 0
province_name        0
latitude             0
longitude            0
baths                0
area                 0
purpose              0
bedrooms             0
date_added           0
agency           44071
agent            44072
Area Type            0
Area Size            0
Area Category        0
dtype: int64

In [4]:
raw['purpose'].value_counts(), raw['property_type'].value_counts(), raw['city'].value_counts()

(purpose
 For Sale    120655
 For Rent     47791
 Name: count, dtype: int64,
 property_type
 House            105468
 Flat              38238
 Upper Portion     13774
 Lower Portion      9229
 Room                685
 Farm House          657
 Penthouse           395
 Name: count, dtype: int64,
 city
 Karachi       60484
 Lahore        41427
 Islamabad     37426
 Rawalpindi    20976
 Faisalabad     8133
 Name: count, dtype: int64)

## 2. Cleaning

In [5]:
df_clean, log = clean(raw, return_log=True)
pd.DataFrame(log, columns=["step", "rows left"])

,step,rows left
0,Raw listings (sale + rent),168446
1,"Keep 'For Sale' only (rent prices are monthly,...",120655
2,Keep residential types (drop single 'Room' lis...,120630
3,Remove duplicate listings (same property poste...,82307
4,Bedrooms and baths between 1 and 12 (0 = not f...,63766
5,"Area between 1 marla and 10 kanal, price > 0",63642
6,GPS coordinates inside the listed city,63625
7,Remove extreme price-per-marla (1st / 99th per...,62379


![cleaning funnel](results/figures/cleaning_funnel.png)

* **For Sale only:** rent is a monthly price, which is a different target.
* **Duplicates removed:** the same property posted several times would otherwise appear in both train and test.
* **0 bedrooms or baths** means the field was left empty, so those rows are dropped.
* **Price per marla outliers** (typos) are removed within each city.

In [6]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
property_id,"62,379.0000","14,813,670.8134","2,532,637.3491","86,575.0000","13,419,925.0000","15,679,275.0000","16,835,430.0000","17,357,652.0000"
latitude,"62,379.0000",29.5698,3.6866,24.7494,24.9361,31.4433,33.4856,33.8246
longitude,"62,379.0000",71.2795,3.2586,66.8420,67.1193,73.0747,74.2630,74.5015
area_marla,"62,379.0000",9.6410,9.6419,1.0000,4.8000,6.7000,10.0000,200.0000
bedrooms,"62,379.0000",3.9272,1.6818,1.0000,3.0000,4.0000,5.0000,12.0000
baths,"62,379.0000",4.0726,1.8329,1.0000,3.0000,4.0000,5.0000,12.0000
price,"62,379.0000","24,331,696.6287","34,398,245.1136","700,000.0000","7,350,000.0000","14,000,000.0000","27,000,000.0000","900,000,000.0000"
listed_by_agency,"62,379.0000",0.6031,0.4893,0.0000,0.0000,1.0000,1.0000,1.0000


## 3. Exploratory data analysis

![price](results/figures/eda_price_distribution.png)

![city/type](results/figures/eda_city_and_type.png)

![maps](results/figures/eda_city_maps.png)

![neighbourhoods](results/figures/eda_top_neighbourhoods.png)

![correlation](results/figures/eda_correlation.png)

**Observations:** prices are very skewed, so the models learn log(price). Area is the strongest predictor. City and
especially the **neighbourhood** shift the price per marla by several times, and rooms and area are strongly correlated.

## 4. Feature engineering, split and preprocessing

In [7]:
df = add_features(df_clean)
df[["area_marla", "log_area_marla", "marla_per_bedroom", "baths_per_bedroom", "total_rooms",
    "distance_to_centre_km", "listed_by_agency", "city_location"]].head()

,area_marla,log_area_marla,marla_per_bedroom,baths_per_bedroom,total_rooms,distance_to_centre_km,listed_by_agency,city_location
0,4.0000,1.3863,2.0000,1.0000,4,4.6240,0,Islamabad | G-10
1,5.6000,1.7228,1.8667,1.0000,6,7.2821,0,Islamabad | E-11
2,8.0000,2.0794,1.6000,1.2000,11,14.2089,0,Islamabad | G-15
3,40.0000,3.6889,10.0000,1.0000,8,9.3795,0,Islamabad | Bani Gala
4,8.0000,2.0794,2.6667,1.0000,6,33.3894,1,Islamabad | DHA Defence


In [8]:
X_train, X_test, y_train, y_test = get_train_test()
print("Features:", FEATURES)
print(f"Train {X_train.shape}  Test {X_test.shape}")

Features: ['area_marla', 'bedrooms', 'baths', 'latitude', 'longitude', 'log_area_marla', 'marla_per_bedroom', 'baths_per_bedroom', 'total_rooms', 'distance_to_centre_km', 'listed_by_agency', 'property_type', 'city', 'city_location']
Train (49903, 14)  Test (12476, 14)


![split](results/figures/data_split.png)

In [9]:
pre = build_preprocessor().fit(X_train, np.log(y_train))
Xt = pre.transform(X_train)
print("Columns after preprocessing:", len(pre.get_feature_names_out()))
print(pre.get_feature_names_out())
print("Neighbourhood target-encoding example (scaled):", Xt[:5, -1].round(3))

Columns after preprocessing: 21
['num__area_marla' 'num__bedrooms' 'num__baths' 'num__latitude'
 'num__longitude' 'num__log_area_marla' 'num__marla_per_bedroom'
 'num__baths_per_bedroom' 'num__total_rooms' 'num__distance_to_centre_km'
 'num__listed_by_agency' 'cat__property_type_Farm House'
 'cat__property_type_Flat' 'cat__property_type_Lower Portion'
 'cat__property_type_Penthouse' 'cat__property_type_Upper Portion'
 'cat__city_Faisalabad' 'cat__city_Islamabad' 'cat__city_Lahore'
 'cat__city_Rawalpindi' 'loc__city_location']
Neighbourhood target-encoding example (scaled): [ 1.546  0.589 -0.101  0.399  0.615]


## 5. Train the six models (log target, metrics in PKR)

In [10]:
best_k = json.load(open("results/summary.json"))["knn_best_k"]   # chosen by 5-fold CV in train.py
models = get_models(knn_k=best_k)
rows, preds = [], {}
for name, model in models.items():
    t = time.perf_counter(); model.fit(X_train, y_train); t = time.perf_counter() - t
    preds[name] = model.predict(X_test)
    r = regression_metrics(y_test, preds[name])
    r["Train R2"] = regression_metrics(y_train, model.predict(X_train))["R2"]
    rows.append({"Model": name, **r, "Train time (s)": t})
    print(f"{name:18s} R2={r['R2']:.4f}  RMSE={format_pkr(r['RMSE']):>12}  MAPE={r['MAPE (%)']:.1f}%  ({t:.1f}s)")

Linear Regression  R2=0.8413  RMSE=  1.38 Crore  MAPE=26.7%  (0.1s)


Decision Tree      R2=0.8842  RMSE=  1.18 Crore  MAPE=22.1%  (0.3s)


Random Forest      R2=0.9117  RMSE=  1.03 Crore  MAPE=20.4%  (2.7s)


Neural Network     R2=0.9045  RMSE=  1.07 Crore  MAPE=21.5%  (23.4s)


XGBoost            R2=0.9178  RMSE=   99.2 Lakh  MAPE=18.2%  (3.1s)


KNN                R2=0.8826  RMSE=  1.19 Crore  MAPE=21.8%  (0.1s)


## 6. Comparative performance table

In [11]:
table = pd.DataFrame(rows).set_index("Model").sort_values("RMSE")
table.insert(0, "Rank", range(1, 7))
show = table.copy()
for c in ("MAE", "RMSE"): show[c] = show[c].map(format_pkr)
show

,Rank,MAE,MSE,RMSE,R2,R2 (log),MAPE (%),Within 10%,Within 20%,Train R2,Train time (s)
Model,,,,,,,,,,,
XGBoost,1,40.7 Lakh,"98,443,382,410,146.5000",99.2 Lakh,0.9178,0.9380,18.2439,40.1811,68.5075,0.9604,3.0874
Random Forest,2,44.1 Lakh,"105,692,184,876,669.0469",1.03 Crore,0.9117,0.9253,20.4202,36.5021,64.3636,0.9245,2.7181
Neural Network,3,46.3 Lakh,"114,407,047,431,924.9219",1.07 Crore,0.9045,0.9194,21.4679,34.1776,61.7506,0.9151,23.3797
Decision Tree,4,48.0 Lakh,"138,687,294,609,384.6562",1.18 Crore,0.8842,0.9110,22.1408,35.4681,61.6784,0.9146,0.3187
KNN,5,48.3 Lakh,"140,559,976,077,003.2188",1.19 Crore,0.8826,0.9142,21.7818,36.6864,62.6723,0.9640,0.0774
Linear Regression,6,59.0 Lakh,"190,022,099,603,376.2500",1.38 Crore,0.8413,0.8861,26.7236,26.7794,50.3607,0.8374,0.1062


In [12]:
cv = pd.read_csv("results/metrics.csv", index_col=0)[["CV RMSE (mean)", "CV RMSE (std)"]]
cv.loc[table.index].map(format_pkr)

,CV RMSE (mean),CV RMSE (std)
Model,,
XGBoost,1.08 Crore,12.1 Lakh
Random Forest,1.14 Crore,14.0 Lakh
Neural Network,1.21 Crore,16.2 Lakh
Decision Tree,1.26 Crore,14.7 Lakh
KNN,1.26 Crore,18.3 Lakh
Linear Regression,1.40 Crore,9.7 Lakh


## 7. Visualisations: actual vs predicted, errors

![actual vs predicted](results/figures/actual_vs_predicted.png)

![samples](results/figures/sample_predictions.png)

![metrics](results/figures/metrics_comparison.png)

![train vs test](results/figures/train_vs_test_r2.png)

![cv](results/figures/cross_validation_rmse.png)

![residuals](results/figures/residuals.png)

![abs error](results/figures/abs_error_boxplot.png)

![by price](results/figures/mape_by_price_range.png)

![by city](results/figures/mape_by_city.png)

![by type](results/figures/mape_by_property_type.png)

![importance](results/figures/feature_importance.png)

![coefficients](results/figures/linear_coefficients.png)

![tree](results/figures/decision_tree_top_levels.png)

![nn](results/figures/nn_training_curve.png)

![knn k](results/figures/knn_k_selection.png)

## 8. Try the best model on a new property

In [13]:
new = pd.DataFrame([dict(property_type="House", city="Lahore", location="Johar Town", latitude=31.469, longitude=74.272,
                         area_marla=10, bedrooms=5, baths=5, listed_by_agency=1)])
xb = add_features(new)[FEATURES]
for n in MODEL_ORDER:
    print(f"{n:18s} {format_pkr(models[n].predict(xb)[0])}")

Linear Regression  2.24 Crore
Decision Tree      2.36 Crore
Random Forest      2.38 Crore
Neural Network     2.21 Crore
XGBoost            2.50 Crore
KNN                2.61 Crore


## 9. Strengths, limitations and conclusion

| Algorithm | Strengths | Limitations |
|---|---|---|
| Linear Regression | Fast, interpretable, good baseline with log(price) and log(area) | Cannot model size × location interactions; under-fits |
| Decision Tree | Non-linear, readable rules | High variance, step-shaped predictions |
| Random Forest | Averaging reduces variance; robust | Bigger model; no extrapolation |
| Neural Network | Smooth non-linear function | Slow, needs tuning and scaling, black box |
| **XGBoost** | Boosting reduces bias, regularisation controls variance, so it has the **best accuracy** | Many hyper-parameters; less interpretable |
| KNN | Intuitive ("similar properties sold for…") | Needs scaling, memorises data, weak on rare types |

In [14]:
best = table.index[0]; r = table.loc[best]
print(f"Best model: {best}")
print(f"  R2 = {r.R2:.4f}, RMSE = {format_pkr(r.RMSE)}, MAE = {format_pkr(r.MAE)}, MAPE = {r['MAPE (%)']:.1f}%")
print(f"  {r['Within 10%']:.1f}% of properties within ±10%, {r['Within 20%']:.1f}% within ±20%")
print(f"  RMSE {100*(1-r.RMSE/table.iloc[1].RMSE):.1f}% lower than {table.index[1]}, "
      f"{100*(1-r.RMSE/table.loc['Linear Regression','RMSE']):.1f}% lower than Linear Regression")

Best model: XGBoost
  R2 = 0.9178, RMSE = 99.2 Lakh, MAE = 40.7 Lakh, MAPE = 18.2%
  40.2% of properties within ±10%, 68.5% within ±20%
  RMSE 3.5% lower than Random Forest, 28.0% lower than Linear Regression


**Conclusion:** XGBoost is the best model on every metric. Property prices depend on interactions between size,
type and location, and gradient-boosted trees model those interactions automatically. Each new tree corrects the
earlier trees' errors (lower bias), while regularisation and shrinkage keep it from over-fitting.